In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc


In [2]:
depress_df = pd.read_csv("Student Depression Dataset.csv")
depress_df.sample(5)

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
22151,111850,Female,29.0,Vadodara,Student,1.0,0.0,6.02,4.0,0.0,Less than 5 hours,Healthy,M.Ed,No,1.0,2.0,No,0
5114,25733,Male,25.0,Varanasi,Student,3.0,0.0,8.74,4.0,0.0,Less than 5 hours,Unhealthy,B.Arch,Yes,12.0,3.0,Yes,0
3933,19698,Male,19.0,Hyderabad,Student,5.0,0.0,6.47,5.0,0.0,Less than 5 hours,Moderate,Class 12,Yes,6.0,5.0,Yes,1
8703,43949,Male,28.0,Ghaziabad,Student,3.0,0.0,9.93,4.0,0.0,More than 8 hours,Healthy,BCA,Yes,4.0,4.0,Yes,0
22500,113458,Male,28.0,Varanasi,Student,5.0,0.0,8.17,3.0,0.0,7-8 hours,Healthy,BE,Yes,5.0,2.0,No,0


https://www.kaggle.com/datasets/hopesb/student-depression-dataset

In [3]:
depress_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     27901 non-null  int64  
 1   Gender                                 27901 non-null  object 
 2   Age                                    27901 non-null  float64
 3   City                                   27901 non-null  object 
 4   Profession                             27901 non-null  object 
 5   Academic Pressure                      27901 non-null  float64
 6   Work Pressure                          27901 non-null  float64
 7   CGPA                                   27901 non-null  float64
 8   Study Satisfaction                     27901 non-null  float64
 9   Job Satisfaction                       27901 non-null  float64
 10  Sleep Duration                         27901 non-null  object 
 11  Di

In [4]:
depress_df.isnull().sum()

id                                       0
Gender                                   0
Age                                      0
City                                     0
Profession                               0
Academic Pressure                        0
Work Pressure                            0
CGPA                                     0
Study Satisfaction                       0
Job Satisfaction                         0
Sleep Duration                           0
Dietary Habits                           0
Degree                                   0
Have you ever had suicidal thoughts ?    0
Work/Study Hours                         0
Financial Stress                         3
Family History of Mental Illness         0
Depression                               0
dtype: int64

In [5]:
depress_df["Financial Stress"] = depress_df["Financial Stress"].fillna(depress_df["Financial Stress"].mean())

In [6]:
depress_df.isnull().sum()

id                                       0
Gender                                   0
Age                                      0
City                                     0
Profession                               0
Academic Pressure                        0
Work Pressure                            0
CGPA                                     0
Study Satisfaction                       0
Job Satisfaction                         0
Sleep Duration                           0
Dietary Habits                           0
Degree                                   0
Have you ever had suicidal thoughts ?    0
Work/Study Hours                         0
Financial Stress                         0
Family History of Mental Illness         0
Depression                               0
dtype: int64

In [7]:
depress_df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,27901.0,70442.149421,40641.175216,2.0,35039.00,70684.00,105818.00,140699.0
Age,27901.0,25.822300,4.905687,18.0,21.00,25.00,30.00,59.0
Academic Pressure,27901.0,3.141214,1.381465,0.0,2.00,3.00,4.00,5.0
Work Pressure,27901.0,0.000430,0.043992,0.0,0.00,0.00,0.00,5.0
CGPA,27901.0,7.656104,1.470707,0.0,6.29,7.77,8.92,10.0
Study Satisfaction,27901.0,2.943837,1.361148,0.0,2.00,3.00,4.00,5.0
Job Satisfaction,27901.0,0.000681,0.044394,0.0,0.00,0.00,0.00,4.0
Work/Study Hours,27901.0,7.156984,3.707642,0.0,4.00,8.00,10.00,12.0
Financial Stress,27901.0,3.139867,1.437269,1.0,2.00,3.00,4.00,5.0
Depression,27901.0,0.585499,0.492645,0.0,0.00,1.00,1.00,1.0


In [8]:
depress_df.describe(include='object').T

,count,unique,top,freq
Gender,27901,2,Male,15547
City,27901,52,Kalyan,1570
Profession,27901,14,Student,27870
Sleep Duration,27901,5,Less than 5 hours,8310
Dietary Habits,27901,4,Unhealthy,10317
Degree,27901,28,Class 12,6080
Have you ever had suicidal thoughts ?,27901,2,Yes,17656
Family History of Mental Illness,27901,2,No,14398


In [9]:
vc = depress_df['City'].value_counts()
mask_vals = vc[vc < 100].index 
depress_df['City'] = depress_df['City'].replace(mask_vals, 'Other')

In [10]:
depress_df['City'].nunique()

31

In [11]:
depress_df['Profession'].value_counts()

Profession
Student                   27870
Architect                     8
Teacher                       6
Digital Marketer              3
Content Writer                2
Chef                          2
Doctor                        2
Pharmacist                    2
Civil Engineer                1
UX/UI Designer                1
Educational Consultant        1
Manager                       1
Lawyer                        1
Entrepreneur                  1
Name: count, dtype: int64

In [12]:
depress_df['Sleep Duration'] = depress_df['Sleep Duration'].replace({'Others':depress_df['Sleep Duration'].mode()[0]})

In [13]:
num_cols = depress_df.select_dtypes(include=["number"]).columns
num_cols

Index(['id', 'Age', 'Academic Pressure', 'Work Pressure', 'CGPA',
       'Study Satisfaction', 'Job Satisfaction', 'Work/Study Hours',
       'Financial Stress', 'Depression'],
      dtype='object')

In [14]:
cat_cols = depress_df.select_dtypes('object').columns
cat_cols

Index(['Gender', 'City', 'Profession', 'Sleep Duration', 'Dietary Habits',
       'Degree', 'Have you ever had suicidal thoughts ?',
       'Family History of Mental Illness'],
      dtype='object')

In [15]:
cols_to_drop = ['id', 'Depression', 'Profession']
X = depress_df.drop(cols_to_drop, axis=1)  # Удаляем таргет
y = depress_df['Depression']

In [16]:
cat_cols = X.select_dtypes('object').columns
num_cols = X.select_dtypes(include=["number"]).columns

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), cat_cols),
        ('num', StandardScaler(), num_cols),
    ], remainder='passthrough')

In [18]:
X_transformed = preprocessor.fit_transform(X)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2, random_state=42)

In [20]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.82      0.79      0.80      2343
           1       0.85      0.88      0.86      3238

    accuracy                           0.84      5581
   macro avg       0.84      0.83      0.83      5581
weighted avg       0.84      0.84      0.84      5581

0.8387385773158932


In [21]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 334797 stored elements and shape (22320, 81)>

In [23]:
# Вставьте этот код в новую ячейку после ячейки с X_transformed = preprocessor.fit_transform(X)

from sklearn.decomposition import PCA

# Применение метода главных компонент (PCA) для снижения размерности
# Сохраняем 20 компонент
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_transformed.toarray())  # Преобразуем в dense, если X_transformed sparse

# Обновляем train_test_split с X_pca
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

In [24]:
X_train.shape

(22320, 43)

In [25]:
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.82      0.79      0.80      2343
           1       0.85      0.88      0.86      3238

    accuracy                           0.84      5581
   macro avg       0.84      0.83      0.83      5581
weighted avg       0.84      0.84      0.84      5581

0.8385593979573553


In [97]:
y_pred_proba = rf_gini_pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Построение графика
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC-кривая (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Случайная модель')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC-кривая')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

NameError: name 'rf_gini_pipeline' is not defined

In [ ]:
accuracy_score(y_test, y_pred)

0.8315714029743774

In [ ]:
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted_RF': y_pred,
    'MAE': np.abs(y_test - y_pred),
})
results.to_csv('depression_predictions.csv', index=False)
results.head()

,Actual,Predicted_RF,MAE
19981,0,0,0
16551,0,0,0
7640,0,0,0
21266,1,1,0
15759,1,0,1


In [ ]:
mean_absolute_error(y_test, y_pred)

0.16842859702562266

In [ ]:
grades_df = pd.read_csv("../datasets/data.csv")
cols_to_drop = ['Grades']
X = grades_df.drop(cols_to_drop, axis=1)  # Удаляем таргет
y = grades_df['Grades']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_reg = RandomForestRegressor()
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

In [ ]:
def print_metrics(y_true, y_pred, model_name):
    print(f'{model_name}:')
    print(f'R2: {r2_score(y_true, y_pred):.4f}')
    print(f'MAE: {mean_absolute_error(y_true, y_pred):.4f}')
    print(f'MSE: {mean_squared_error(y_true, y_pred):.4f}\n')

print_metrics(y_test, y_pred_rf, 'Случайный лес')


Случайный лес:
R2: 0.9811
MAE: 0.8973
MSE: 1.4295

